# Submission 13: Experiment 41D 5-Fold Rank Blend

Reproduces the Experiment 41D ensemble for the Kaggle test set.

- 5-fold 33B XGBoost
- 5-fold Experiment 39 CatBoost
- Average test predictions across folds
- Rank-transform both model predictions
- Final blend: 45% CatBoost rank + 55% XGBoost rank
- Output: `../submissions/submission_13.csv`



In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
OUTPUT_PATH = "../submissions/submission_13_v2.csv"

TARGET = "Will_Buy_EV"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

X = train.drop(columns=[TARGET, "id"]).copy()
X_test = test.drop(columns=["id"]).copy()

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Train rows:", len(X))
print("Test rows:", len(X_test))
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


# ============================================================
# 1. EXACT 33B IDENTITY ENCODING
# ============================================================

def make_identity_key(series):
    return series.astype("string").fillna("__MISSING__")


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        "value": values,
        "target": target.to_numpy()
    })

    global_mean = float(target.mean())

    stats = (
        temp.groupby("value", dropna=False)["target"]
        .agg(["mean", "count"])
    )

    smoothed = (
        stats["count"] * stats["mean"]
        + smoothing * global_mean
    ) / (stats["count"] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return (
        values.map(mapping)
        .fillna(global_mean)
        .astype(float)
    )


def add_identity_features(X_fit, y_fit, X_apply, columns,
                          n_splits=3, smoothing=20):

    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf_inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:

        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf_inner.split(X_fit, y_fit):

            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = apply_mapping(
                fit_keys.iloc[fold_idx],
                mapping,
                global_mean
            ).to_numpy()

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[f"{col}__identity_target"] = oof_values

        X_apply[f"{col}__identity_target"] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

        frequencies = fit_keys.value_counts(dropna=False)

        X_fit[f"{col}__identity_frequency"] = (
            fit_keys.map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

        X_apply[f"{col}__identity_frequency"] = (
            apply_keys.map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

    return X_fit, X_apply


# ============================================================
# 2. EXACT 33B DIGIT DECOMPOSITION
# ============================================================

def add_digit_features(X_frame, columns):

    X_frame = X_frame.copy()

    for col in columns:

        values = pd.to_numeric(
            X_frame[col],
            errors="coerce"
        )

        integer_values = values.abs().round()

        X_frame[f"{col}__digits"] = (
            np.floor(
                np.log10(
                    integer_values.clip(lower=1)
                )
            ) + 1
        )

        divisor = 10 ** (
            X_frame[f"{col}__digits"] - 1
        )

        X_frame[f"{col}__first_digit"] = (
            integer_values / divisor
        ).fillna(0).astype(float)

        X_frame[f"{col}__first_digit"] = np.floor(
            X_frame[f"{col}__first_digit"]
        )

        X_frame[f"{col}__last_digit"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 10
        )

        def digit_sum(v):
            if pd.isna(v):
                return np.nan

            s = str(int(abs(v)))
            return sum(int(ch) for ch in s)

        X_frame[f"{col}__digit_sum"] = (
            integer_values.map(digit_sum)
        )

        X_frame[f"{col}__parity"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 2
        )

        X_frame[f"{col}__mod100"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 100
        )

        X_frame[f"{col}__mod1000"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 1000
        )

        X_frame[f"{col}__ends_zero"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 10 == 0
        ).astype(np.int8)

    return X_frame


# ============================================================
# 3. XGBOOST PREPROCESSOR
# ============================================================

def build_preprocessor(X_frame):

    numeric = X_frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical = X_frame.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric),
        ("cat", categorical_pipeline, categorical)
    ])


# ============================================================
# 4. EXACT 33B 5-FOLD TEST PREDICTIONS
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_test_preds = []

print("")
print("=" * 70)
print("33B XGBOOST 5-FOLD TEST PREDICTIONS")
print("=" * 70)

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(X, y),
    start=1
):

    print("")
    print("-" * 60)
    print(f"33B Fold {fold}")
    print("-" * 60)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]

    X_te = X_test.copy()

    X_tr_id, X_te_id = add_identity_features(
        X_tr,
        y_tr,
        X_te,
        numeric_cols,
        n_splits=3,
        smoothing=20
    )

    X_tr_digit = add_digit_features(
        X_tr_id,
        numeric_cols
    )

    X_te_digit = add_digit_features(
        X_te_id,
        numeric_cols
    )

    preprocessor = build_preprocessor(X_tr_digit)

    X_tr_encoded = preprocessor.fit_transform(
        X_tr_digit
    )

    X_te_encoded = preprocessor.transform(
        X_te_digit
    )

    print("Encoded shape:", X_tr_encoded.shape)

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr_encoded,
        y_tr,
        verbose=False
    )

    pred = model.predict_proba(
        X_te_encoded
    )[:, 1]

    xgb_test_preds.append(pred)

    print(
        f"Fold {fold} mean prediction: "
        f"{pred.mean():.6f}"
    )


xgb_test_pred = np.mean(
    np.vstack(xgb_test_preds),
    axis=0
)

print("")
print(
    f"XGB averaged test prediction mean: "
    f"{xgb_test_pred.mean():.6f}"
)


# ============================================================
# 5. EXACT EXPERIMENT 39 CATBOOST FEATURES
# ============================================================

cat_cols = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

subsidy = (
    X["Subsidy_Available"]
    .astype("string")
    .fillna("__MISSING__")
    .eq("Yes")
    .astype(int)
)

home = (
    X["Home_Charging_Possible"]
    .astype("string")
    .fillna("__MISSING__")
    .eq("Yes")
    .astype(int)
)

X_cat = X.copy()
X_test_cat = X_test.copy()

X_cat["Subsidy_x_EnvConcern"] = (
    subsidy * X_cat["Environmental_Concern_Level"]
)

X_cat["Subsidy_x_Income"] = (
    subsidy * X_cat["Annual_Income_USD"]
)

X_cat["Subsidy_x_HomeCharging"] = (
    subsidy * home
)

subsidy_test = (
    X_test_cat["Subsidy_Available"]
    .astype("string")
    .fillna("__MISSING__")
    .eq("Yes")
    .astype(int)
)

home_test = (
    X_test_cat["Home_Charging_Possible"]
    .astype("string")
    .fillna("__MISSING__")
    .eq("Yes")
    .astype(int)
)

X_test_cat["Subsidy_x_EnvConcern"] = (
    subsidy_test *
    X_test_cat["Environmental_Concern_Level"]
)

X_test_cat["Subsidy_x_Income"] = (
    subsidy_test *
    X_test_cat["Annual_Income_USD"]
)

X_test_cat["Subsidy_x_HomeCharging"] = (
    subsidy_test * home_test
)

value_identity_cols = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work"
]

for col in value_identity_cols:

    X_cat[f"{col}__value_id"] = (
        X_cat[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    X_test_cat[f"{col}__value_id"] = (
        X_test_cat[col]
        .astype("string")
        .fillna("__MISSING__")
    )


cat_identity_cols = cat_cols + [
    f"{c}__value_id"
    for c in value_identity_cols
]

for col in cat_identity_cols:

    X_cat[col] = (
        X_cat[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    X_test_cat[col] = (
        X_test_cat[col]
        .astype("string")
        .fillna("__MISSING__")
    )


# ============================================================
# 6. EXACT EXPERIMENT 39 CATBOOST 5-FOLD TEST PREDICTIONS
# ============================================================

cat_test_preds = []

print("")
print("=" * 70)
print("CATBOOST 5-FOLD TEST PREDICTIONS")
print("=" * 70)

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(X_cat, y),
    start=1
):

    print("")
    print("-" * 60)
    print(f"CatBoost Fold {fold}")
    print("-" * 60)

    X_tr = X_cat.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        l2_leaf_reg=3,
        random_strength=1,
        bootstrap_type="Bayesian",
        bagging_temperature=1,
        random_seed=7,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_identity_cols,
        verbose=False
    )

    pred = model.predict_proba(
        X_test_cat
    )[:, 1]

    cat_test_preds.append(pred)

    print(
        f"Fold {fold} mean prediction: "
        f"{pred.mean():.6f}"
    )


cat_test_pred = np.mean(
    np.vstack(cat_test_preds),
    axis=0
)

print("")
print(
    f"CatBoost averaged test prediction mean: "
    f"{cat_test_pred.mean():.6f}"
)


# ============================================================
# 7. EXPERIMENT 41D RANK BLEND
# ============================================================

cat_rank = (
    pd.Series(cat_test_pred)
    .rank(method="average", pct=True)
    .to_numpy()
)

xgb_rank = (
    pd.Series(xgb_test_pred)
    .rank(method="average", pct=True)
    .to_numpy()
)

CAT_WEIGHT = 0.45
XGB_WEIGHT = 0.55

final_pred = (
    CAT_WEIGHT * cat_rank
    + XGB_WEIGHT * xgb_rank
)


# ============================================================
# 8. CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "id": test["id"],
    TARGET: final_pred
})

assert len(submission) == len(test)
assert list(submission.columns) == ["id", TARGET]
assert submission["id"].equals(test["id"])
assert submission[TARGET].notna().all()
assert ((submission[TARGET] >= 0) & (submission[TARGET] <= 1)).all()

Path("../submissions").mkdir(
    parents=True,
    exist_ok=True
)

submission.to_csv(
    OUTPUT_PATH,
    index=False
)

print("")
print("=" * 70)
print("SUBMISSION 13 CREATED")
print("=" * 70)
print(f"Path: {OUTPUT_PATH}")
print(f"Rows: {len(submission)}")
print(f"Columns: {list(submission.columns)}")
print(f"Prediction mean: {final_pred.mean():.6f}")
print(f"Prediction min: {final_pred.min():.6f}")
print(f"Prediction max: {final_pred.max():.6f}")
print("")
print("Experiment 41D weights:")
print("CatBoost rank: 45%")
print("XGBoost rank: 55%")
print("")


Train rows: 668665
Test rows: 286571
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

33B XGBOOST 5-FOLD TEST PREDICTIONS

------------------------------------------------------------
33B Fold 1
------------------------------------------------------------
Encoded shape: (534932, 94)
Fold 1 mean prediction: 0.175256

------------------------------------------------------------
33B Fold 2
------------------------------------------------------------
Encoded shape: (534932, 94)
Fold 2 mean prediction: 0.175159

------------------------------------------------------------
33B Fold 3
------------------------------------------------------------
Encoded shape: (534932, 94)
Fold 3 mean prediction: 0.174988

----------------------

KeyboardInterrupt: 